# CNN Training — MNIST-style Handwritten Digits

Notebook này huấn luyện CNN PyTorch trên dữ liệu gốc của project. Không dùng torchvision; ảnh PNG được đọc bằng Pillow và nhãn được lấy từ tên file.

Pipeline: `PNG → grayscale → 28×28 → tensor [0,1] → MNISTCNN → logits → CrossEntropyLoss`.

Chạy notebook từ thư mục gốc project. Notebook chỉ train khi người dùng chạy các cell training.

## 1. Imports and configuration

Chốt seed, đường dẫn dữ liệu, hyperparameters và device trước khi train.

In [1]:
from pathlib import Path
import random
import re
import sys

import numpy as np
import torch
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset

MODEL_DIR = Path.cwd() / "Model" / "CNN model"
PROJECT_ROOT = MODEL_DIR.parents[1]
DATA_DIR = PROJECT_ROOT / "Model" / "Dataset" / "Raw data"
CHECKPOINT_DIR = MODEL_DIR / "artifacts"
CHECKPOINT_PATH = CHECKPOINT_DIR / "mnist_cnn.pt"
SEED = 42
BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 1e-3
NUM_WORKERS = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

sys.path.insert(0, str(MODEL_DIR))
from cnn_model import MNISTCNN

print({"data_dir": str(DATA_DIR), "device": str(DEVICE), "epochs": EPOCHS})

{'data_dir': 'd:\\AL_HK3\\Model\\CNN model\\Model\\Dataset\\Raw data', 'device': 'cuda', 'epochs': 10}


## 2. Load the original image dataset

Dữ liệu phải tồn tại ở `Model/Dataset/Raw data`. Không tự tạo dữ liệu thay thế.

In [2]:
if not DATA_DIR.is_dir():
    raise FileNotFoundError(
        f"Missing {DATA_DIR}. Pull/extract the project dataset before training."
    )

image_paths = sorted(DATA_DIR.glob("*.png"))
if not image_paths:
    raise FileNotFoundError(f"No PNG files found in {DATA_DIR}")

def label_from_name(path: Path) -> int:
    match = re.search(r"test_([0-9])_", path.name)
    if match is None:
        raise ValueError(f"Cannot parse digit label from filename: {path.name}")
    return int(match.group(1))

labels = np.asarray([label_from_name(path) for path in image_paths], dtype=np.int64)
print("images:", len(image_paths), "class_counts:", np.bincount(labels, minlength=10))

FileNotFoundError: Missing d:\AL_HK3\Model\CNN model\Model\Dataset\Raw data. Pull/extract the project dataset before training.

## 3. Stratified development split

Giữ validation độc lập để đánh giá model; test set riêng của project không được dùng trong notebook này.

In [ ]:
train_paths, val_paths, train_labels, val_labels = train_test_split(
    image_paths,
    labels,
    test_size=0.2,
    stratify=labels,
    random_state=SEED,
)
print("train:", len(train_paths), "validation:", len(val_paths))

## 4. Dataset and preprocessing

Ảnh được chuyển grayscale, giữ polarity digit sáng/nền tối, resize bảo toàn tỉ lệ và pad thành `28×28`.

In [ ]:
def prepare_image(path: Path) -> torch.Tensor:
    image = Image.open(path).convert("L")
    image = ImageOps.contain(image, (28, 28), Image.Resampling.LANCZOS)
    canvas = Image.new("L", (28, 28))
    canvas.paste(image, ((28 - image.width) // 2, (28 - image.height) // 2))
    array = np.asarray(canvas, dtype=np.float32) / 255.0
    border = np.concatenate((array[0], array[-1], array[:, 0], array[:, -1]))
    if border.mean() > 0.5:
        array = 1.0 - array
    return torch.from_numpy(array).unsqueeze(0)

class DigitDataset(Dataset):
    def __init__(self, paths, labels):
        self.paths = list(paths)
        self.labels = torch.as_tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        return prepare_image(self.paths[index]), self.labels[index]

train_loader = DataLoader(
    DigitDataset(train_paths, train_labels),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
)
val_loader = DataLoader(
    DigitDataset(val_paths, val_labels),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)


## 5. Training and evaluation functions

Cell này chỉ định nghĩa logic; chưa train model.

In [ ]:
def run_epoch(model, loader, loss_fn, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    correct = 0
    total = 0
    for images, labels_batch in loader:
        images, labels_batch = images.to(DEVICE), labels_batch.to(DEVICE)
        if training:
            optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = loss_fn(logits, labels_batch)
        if training:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * labels_batch.size(0)
        correct += (logits.argmax(1) == labels_batch).sum().item()
        total += labels_batch.size(0)
    return total_loss / total, correct / total


## 6. Train CNN

Chạy cell này để train. Đây là cell có side effect tạo checkpoint.

In [ ]:
model = MNISTCNN().to(DEVICE)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, loss_fn)
    history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc, "val_loss": val_loss, "val_acc": val_acc})
    print(history[-1])

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
torch.save({"model_state_dict": model.state_dict(), "config": {"seed": SEED, "epochs": EPOCHS, "learning_rate": LEARNING_RATE}}, CHECKPOINT_PATH)
print("saved:", CHECKPOINT_PATH)

## 7. Final validation report

Đánh giá checkpoint/model hiện tại trên validation; không dùng kết quả này để giả lập test.

In [ ]:
val_loss, val_acc = run_epoch(model, val_loader, loss_fn)
print({"validation_loss": val_loss, "validation_accuracy": val_acc})

## 8. Reproducible checkpoint loading

Dùng cell này khi inference cần nạp model đã train.

In [ ]:
def load_checkpoint(path=CHECKPOINT_PATH, device=DEVICE):
    checkpoint = torch.load(path, map_location=device, weights_only=True)
    loaded = MNISTCNN().to(device)
    loaded.load_state_dict(checkpoint["model_state_dict"])
    loaded.eval()
    return loaded, checkpoint.get("config", {})
